# Using Prompt for Base Model

In [1]:
# dependencies
%pip install datasets -q transformers accelerate torch

In [2]:
# Preview Datasets
from datasets import load_dataset

# Generate JSON Output
import json

# Load model
import torch
import re
from transformers import pipeline

In [3]:
# Configurations

# Dataset
DATASETS = {
    "gsm8k": ("gsm8k", "main"),
    "asdiv": ("yimingzhang/asdiv", None),
    "metamathqa": ("meta-math/MetaMathQA", None),
    "openmathinstruct": ("nvidia/OpenMathInstruct-1", None), # Can change to nvidia/OpenMathInstruct-2 for larger datasets
}

DATASET_CONFIG = {
    "gsm8k": {
        "path": "gsm8k",
        "subset": "main",
        "split": "test",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["answer"]
    },
    "asdiv": {
        "path": "yimingzhang/asdiv",
        "subset": None,
        "split": "test",
        "extract_question": lambda x: x["text"].replace("Question:", "").replace("Answer:", "").strip(),
        "extract_answer": lambda x: x["label"]   # clean numeric answer
    },
    "metamathqa": {
        "path": "meta-math/MetaMathQA",
        "subset": None,
        "split": "train", # No testing dataset available
        "extract_question": lambda x: x["original_question"],
        "extract_answer": lambda x: x["response"]
    },
    "openmathinstruct": {
        "path": "nvidia/OpenMathInstruct-1", # Can change to nvidia/OpenMathInstruct-2 for larger datasets
        "subset": None,
        "split": "validation",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["expected_answer"]
    }
}

# Model
MODEL_NAME = "Qwen/Qwen2-1.5B"

DEVICE = 0 if torch.cuda.is_available() else -1

TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

In [4]:
# Key Functions

# PROMPT
# Build Prompt
def build_prompt(question, name):
    return f"""
You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step using clear logical reasoning.
Show all intermediate reasoning clearly.
At the end, provide the final numerical answer.

Follow these steps:
1. Analyze the problem.
2. Compute step by step.
3. Double-check calculations.
4. Provide final result.

Important rules:
- Do NOT include any text outside the JSON.
- Do NOT use markdown.
- Ensure the final answer is a single number or expression.
- Do NOT restate the instructions.

Output your response strictly in the following JSON format:
{{
  "dataset": "{name}",
  "question": "...",
  "model_reasoning": "...",
  "model_answer": "..."
}}

Problem:
{question}
"""

# MODEL
# Input prompt to model
def ask_model(prompt, name, question, answer,max_tokens=256):
    output = pipe(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    raw_response = output[0]["generated_text"].strip()
    
    json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw_response)
    if json_match:
        json_str = json_match.group(0)
    else:
        # Fallback: find first complete JSON object
        start_idx = raw_response.find('{')
        end_idx = raw_response.rfind('}') + 1
        json_str = raw_response[start_idx:end_idx] if start_idx != -1 else '{}'
    
    try:
        model_output = json.loads(json_str)
    except json.JSONDecodeError:
        # Ultimate fallback: return empty structure
        model_output = {"model_reasoning": "", "model_answer": raw_response}
    
    # Build ideal structure (override dataset/question with actual values for safety)
    response = {
        "dataset": name,
        "question": question,
        "model_reasoning": model_output.get("model_reasoning", ""),
        "model_answer": model_output.get("model_answer", raw_response),
        "correct_answer": answer
    }
    
    return response

def extract_number(text):
    numbers = re.findall(r"\d+", text)
    if numbers:
        return int(numbers[-1])
    return None


def evaluate_math_question(question, correct_answer):
    response = ask_model(question)
    predicted = extract_number(response)

    print("Question:", question)
    print("Model Response:", response)
    print("Extracted Answer:", predicted)
    print("Correct Answer:", correct_answer)

    if predicted == correct_answer:
        print("✅ Correct\n")
        return True
    else:
        print("❌ Incorrect\n")
        return False

In [5]:
for name, (path, subset) in DATASETS.items():
    print("\n" + "=" * 70)
    print(f"Dataset: {name}")
    print("=" * 70)

    try:
        # Load dataset
        if subset:
            dataset = load_dataset(path, subset)
        else:
            dataset = load_dataset(path)

        print("Available splits:", list(dataset.keys()))

        for split in dataset.keys():
            print(f"\n--- Split: {split} ---")
            print("Number of samples:", len(dataset[split]))

            # Column names
            print("Column names:", dataset[split].column_names)

            # Feature types (schema)
            print("Features:")
            print(dataset[split].features)

            # Print one example
            print("\nFirst sample:")
            print(dataset[split][0])

    except Exception as e:
        print("Error loading dataset:", e)


Dataset: gsm8k


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Available splits: ['train', 'test']

--- Split: train ---
Number of samples: 7473
Column names: ['question', 'answer']
Features:
{'question': Value('string'), 'answer': Value('string')}

First sample:
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

--- Split: test ---
Number of samples: 1319
Column names: ['question', 'answer']
Features:
{'question': Value('string'), 'answer': Value('string')}

First sample:
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 1

Available splits: ['train', 'test']

--- Split: train ---
Number of samples: 1200
Column names: ['text', 'target', 'label']
Features:
{'text': Value('string'), 'target': Value('string'), 'label': Value('string')}

First sample:
{'text': 'Question: Rachel bought 8 music albums online. If each album had 2 songs, how many songs did she buy total?\nAnswer:', 'target': '<<8*2>>16', 'label': '16'}

--- Split: test ---
Number of samples: 301
Column names: ['text', 'target', 'label']
Features:
{'text': Value('string'), 'target': Value('string'), 'label': Value('string')}

First sample:
{'text': 'Question: Sam had eleven friends over for his birthday party. Later three of his friends had to go home. How many friends were left?\nAnswer:', 'target': '<<11-3>>8', 'label': '8'}

Dataset: metamathqa
Available splits: ['train']

--- Split: train ---
Number of samples: 395000
Column names: ['type', 'query', 'original_question', 'response']
Features:
{'type': Value('string'), 'query': Value('string'), 

In [6]:
pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
all_outputs = []

for name, config in DATASET_CONFIG.items():

    print(f"Processing {name}...")

    dataset = load_dataset(
        config["path"],
        config["subset"] if config["subset"] else None
    )

    split_data = dataset[config["split"]]

    for sample in split_data:

        question = config["extract_question"](sample)
        answer = config.get("extract_answer", lambda x: None)(sample)        
        prompt = build_prompt(question, name)

        # ---- send to your model here ----
        response = ask_model(prompt, name, question, answer) # previous code: model.generate(prompt)

        # # For demonstration only:
        # response = {
        #     "dataset": name,
        #     "question": question,
        #     "model_reasoning": "model reasoning here",
        #     "model_answer": "model answer",
        #     "correct_answer": answer
        # }

        all_outputs.append(response)

        # Optional: limit samples per dataset
        if len(all_outputs) >= 1000:
            break

In [ ]:
# Save to JSON
with open("all_outputs.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

# Past Test Cases

In [7]:
# MAIN TEST CASE
question = "If I have 5 red apples and 7 green apples, how many apples do I have?"

prompt = build_prompt(question, "gsm8k")

response = ask_model(prompt, "gsm8k", question, "12") # Providing correct answer for evaluation

print("Model Output:")
print(response)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model Output:
{'dataset': 'gsm8k', 'question': 'If I have 5 red apples and 7 green apples, how many apples do I have?', 'model_reasoning': 'To find the total number of apples, we can add the number of red apples to the number of green apples.', 'model_answer': 'There are 12 apples in total.', 'correct_answer': '12'}


In [ ]:
print(type(response))

In [ ]:
print(prompt)

In [ ]:
# Test Case 1
question = "If I have 5 red apples and 7 green apples, how many apples do I have?"

response = ask_model(question)

print("Model Output:")
print(response)

In [ ]:
questions = [
    ("If I have 5 red apples and 7 green apples, how many apples do I have?", 12),
    ("John has 10 books and buys 3 more. How many books does he have?", 13),
    ("There are 20 birds, 5 fly away. How many remain?", 15),
    ("Sarah has 8 candies and eats 2. How many are left?", 6),
]

correct = 0

for q, ans in questions:
    if evaluate_math_question(q, ans):
        correct += 1

accuracy = (correct / len(questions))*100
print(f"Final Accuracy: {accuracy:.2f}")